In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
MODEL = os.getenv("MODEL", "gpt-5")

assert OPENAI_API_KEY, "Missing OPENAI_API_KEY in .env"

In [2]:
import numpy as np
from dataclasses import dataclass
from typing import List
import fitz  # PyMuPDF
import tiktoken
from openai import OpenAI

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ======== CONFIG (edit these) ========
PDF_PATH = r"C:\Users\Martin Snyman\Desktop\Work\tax_guide_pdf.pdf"
# EMBED_CACHE = "cache/embeddings.npz"     # where the cache is saved
EMBED_MODEL = "text-embedding-3-small"   # OpenAI embeddings model
CHUNK_TOKENS = 400                        # chunk size (approx. tokens)
CHUNK_OVERLAP = 60                        # overlap between chunks
# =====================================

# Load environment & client
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Missing OPENAI_API_KEY in environment/.env"
client = OpenAI(api_key=OPENAI_API_KEY)

# Encoder (robust fallback)
def _get_encoder():
    try:
        return tiktoken.encoding_for_model("gpt-4o-mini")
    except Exception:
        return tiktoken.get_encoding("cl100k_base")
ENC = _get_encoder()

def read_pdf_text_pymupdf(path: str) -> str:
    """Extract text from a PDF using PyMuPDF only."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"PDF not found: {path}")
    parts: List[str] = []
    with fitz.open(path) as doc:
        for page in doc:
            # "text" gives readable layout text; you can try "block" if you want more structure
            parts.append(page.get_text("text"))
    return "\n\n".join(parts).strip()

def chunk_text_by_tokens(text: str, target_tokens=CHUNK_TOKENS, overlap=CHUNK_OVERLAP) -> List[str]:
    """Token-based chunking using tiktoken."""
    text = (text or "").strip()
    if not text:
        return []
    ids = ENC.encode(text)
    if not ids:
        return []
    chunks: List[str] = []
    start = 0
    while start < len(ids):
        end = min(start + target_tokens, len(ids))
        chunk_ids = ids[start:end]
        chunks.append(ENC.decode(chunk_ids))
        if end == len(ids):
            break
        start = max(0, end - overlap)
    return chunks

def embed_texts(texts: List[str]) -> np.ndarray:
    """Embed a list of strings with OpenAI embeddings API."""
    if not texts:
        return np.zeros((0, 1536), dtype=np.float32)
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)

@dataclass
class PdfIndex:
    chunks: List[str]
    matrix: np.ndarray


In [3]:

CHROMA_DIR = "chroma/pdf_demo"       # persisted local store
TOP_K = 5                            # default; you can expose a slider later

# def read_pdf_as_docs(PDF_PATH: str) -> list[Document]:
#     assert os.path.exists(PDF_PATH), f"PDF not found: {PDF_PATH}"
#     docs = []
#     with fitz.open(PDF_PATH) as doc:
#         for i, page in enumerate(doc, start=1):
#             text = page.get_text("text") or ""
#             if text.strip():
#                 docs.append(Document(page_content=text, metadata={"source": os.path.basename(PDF_PATH), "page": i}))
#     return docs

def _is_roman_label(s: str) -> bool:
    return bool(re.fullmatch(r"[ivxlcdm]+", s.strip().lower()))

def detect_frontmatter_roman_count(pdf_path: str, max_check: int = 30) -> int:
    """Try to auto-detect how many initial pages are Roman-numbered using PDF page labels.
       Falls back to 0 if labels not present / not roman."""
    try:
        with fitz.open(pdf_path) as doc:
            count = 0
            for i in range(min(max_check, len(doc))):
                pg = doc[i]
                label = ""
                try:
                    # PyMuPDF provides page labels on newer versions:
                    # pg.get_label() or doc.get_page_labels(); using try/except to be safe.
                    label = pg.get_label()  # type: ignore[attr-defined]
                except Exception:
                    label = ""
                if label and _is_roman_label(label):
                    count += 1
                else:
                    break
            return count
    except Exception:
        return 0

def read_pdf_as_docs(pdf_path: str, frontmatter_roman_count: int | None = None) -> list[Document]:
    assert os.path.exists(pdf_path), f"PDF not found: {pdf_path}"
    if frontmatter_roman_count is None:
        frontmatter_roman_count = detect_frontmatter_roman_count(pdf_path)

    docs = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc, start=1):
            text = page.get_text("text") or ""
            if not text.strip():
                continue
            logical = i - frontmatter_roman_count if i > frontmatter_roman_count else None
            # Also try to grab the PDF’s own label, if available
            label = ""
            try:
                label = page.get_label()  # type: ignore[attr-defined]
            except Exception:
                label = ""

            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": os.path.basename(pdf_path),
                        "page_physical": i,          # physical page index starting at 1
                        "page_label": label,         # e.g., i, ii, 1, 2, 3...
                        "logical_page": logical      # None for Roman pages, 1..N for main body
                    },
                )
            )
    return docs



splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,   # ≈ 350–450 tokens
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)

# def build_vectorstore(PDF_PATH: str, persist_dir: str = CHROMA_DIR):
#     # 1) extract per-page → Documents
#     page_docs = read_pdf_as_docs(PDF_PATH)
#     # 2) split into chunks, preserve page metadata
#     docs = []
#     for d in page_docs:
#         for c in splitter.split_text(d.page_content):
#             docs.append(Document(page_content=c, metadata=d.metadata))
#     # 3) embed + persist Chroma
#     embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))
#     vs = Chroma.from_documents(docs, embedding=embeddings, persist_directory=persist_dir)
#     vs.persist()
#     return vs

# def load_or_build_vectorstore(persist_dir: str = CHROMA_DIR):
#     embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))
#     if os.path.exists(os.path.join(persist_dir, "chroma.sqlite")):
#         return Chroma(embedding_function=embeddings, persist_directory=persist_dir)
#     else:
#         return build_vectorstore(PDF_PATH, persist_dir)

def build_vectorstore(pdf_path: str, persist_dir: str = CHROMA_DIR):
    page_docs = read_pdf_as_docs(pdf_path)  # auto-detects roman front matter
    chunked = []
    for d in page_docs:
        for chunk in splitter.split_text(d.page_content):
            chunked.append(
                Document(page_content=chunk, metadata=d.metadata)
            )
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))
    vs = Chroma.from_documents(chunked, embedding=embeddings, persist_directory=persist_dir)
    vs.persist()
    return vs

def load_or_build_vectorstore(persist_dir: str = CHROMA_DIR):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))
    if os.path.exists(os.path.join(persist_dir, "chroma.sqlite")):
        return Chroma(embedding_function=embeddings, persist_directory=persist_dir)
    else:
        return build_vectorstore(PDF_PATH, persist_dir)
    

import shutil
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print("🧹 Deleted old Chroma dir.")    

vectorstore = load_or_build_vectorstore()
print("Vector store docs:", vectorstore._collection.count())



🧹 Deleted old Chroma dir.
Vector store docs: 314


C:\Users\Martin Snyman\AppData\Local\Temp\ipykernel_1972\1715068994.py:111: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vs.persist()


In [4]:
#Page Specific Retrieval


import re

def get_page_number_from_question(q: str) -> int | None:
    """
    Extracts an integer page number when the user asks like:
    'summarize page 71', 'page 12 please', etc.
    """
    m = re.search(r"\bpage\s+(\d{1,4})\b", q.strip().lower())
    return int(m.group(1)) if m else None

def retrieve_page_docs(page_num: int, k: int = 12) -> List[Document]:
    """
    Return up to k chunks ONLY from the specified page number using metadata filtering.
    Works with recent LangChain+Chroma (filter=...). If your LC version is older,
    it may require 'where=' instead of 'filter=' (see fallback).
    """
    try:
        # Newer LC uses 'filter'
        docs = vectorstore.similarity_search(
            query=".",   # dummy; filter restricts results
            k=k,
            filter={"page": page_num}
        )
        return docs
    except TypeError:
        # Fallback for older LC versions that expect 'where'
        docs = vectorstore.similarity_search(
            query=".",
            k=k,
            where={"page": page_num}  # type: ignore
        )
        return docs

def format_docs_with_citations(docs: List[Document]) -> str:
    """
    Render docs with page citations for the prompt.
    """
    parts = []
    for d in docs:
        src = d.metadata.get("source", "PDF")
        pg = d.metadata.get("page", "?")
        parts.append(f"[{src} p.{pg}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


In [5]:
# LLM (GPT-5 via LangChain)
llm = ChatOpenAI(
    model=os.getenv("MODEL", "gpt-5"),
    temperature=0.2,
    api_key=os.getenv("OPENAI_API_KEY"),
)

# MMR retriever for diverse chunks
def make_retriever(k: int = TOP_K):
    return vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": k, "fetch_k": max(20, 3*k)})

def format_docs(docs: list[Document]) -> str:
    parts = []
    for d in docs:
        src = d.metadata.get("source", "PDF")
        pg = d.metadata.get("page", "?")
        parts.append(f"[{src} p.{pg}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a precise assistant. Use ONLY the provided context. If missing, say you don't know. Cite pages like (p. 12)."),
        ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
    ]
)

from langchain_core.runnables import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

def make_pdf_chain(k: int = TOP_K):
    retriever = make_retriever(k=k)
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

pdf_chain = make_pdf_chain(TOP_K)

# quick sanity test (optional)
try:
    print(pdf_chain.invoke("Give me a one-sentence summary of this document.")[:300])
except Exception as e:
    print("PDF chain error:", e)


A 2024/2025 Tax Guide for Small Businesses in South Africa that outlines business types, key tax allowances and incentives, compliance with major legislation, and relevant trade arrangements affecting small enterprises (p. iii).


In [6]:

import requests
from typing import Optional


# Tavily Search
def tavily_search(query: str, max_results: int = 5) -> List[dict]:
    """Return list of {title, url, content} from Tavily."""
    if not TAVILY_API_KEY:
        return []
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "search_depth": "advanced",
        "include_answer": False,
        "max_results": max_results
    }
    r = requests.post(url, json=payload, timeout=30)
    if r.ok:
        data = r.json()
        # Tavily returns 'results' list with {title, url, content}
        return data.get("results", [])
    return []

def summarize_tavily_results(results: List[dict], limit_chars: int = 2000) -> str:
    buf = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "")[:120]
        url = r.get("url", "")
        content = (r.get("content", "") or "")[:500]
        buf.append(f"[{i}] {title}\nURL: {url}\nSnippet: {content}\n")
    text = "\n".join(buf)
    return text[:limit_chars]

# Playwright scrape
import sys, asyncio
if sys.platform.startswith("win"):
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        print("Event loop policy set to WindowsProactorEventLoopPolicy.")
    except Exception as e:
        print("Could not set WindowsProactorEventLoopPolicy:", e)
from playwright.sync_api import sync_playwright

def scrape_page(url: str, wait_selector: Optional[str] = None, timeout_ms: int = 10000) -> str:
    """Render a page and return visible text."""
    if not url:
        return ""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        try:
            page = browser.new_page()
            page.goto(url, timeout=timeout_ms)
            if wait_selector:
                page.wait_for_selector(wait_selector, timeout=timeout_ms)
            # Get all visible text
            content = page.inner_text("body")
            return content[:8000]  # keep it light
        finally:
            browser.close()


Event loop policy set to WindowsProactorEventLoopPolicy.


In [7]:
from openai import OpenAI
from typing import List, Tuple

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM = (
    "You are a helpful legal tax assistant. Only use the provided Context sections. "
    "If the answer is not present, say you don't know."
)

def cosine_sim(A: np.ndarray, q: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between each row of A and vector q.
    Returns an array of similarity scores.
    """
    if A.size == 0 or q.size == 0:
        return np.array([])
    # normalize
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    q_norm = q / (np.linalg.norm(q) + 1e-8)
    return A_norm @ q_norm


def call_gpt5(prompt: str) -> str:
    
    resp = client.responses.create(
        model=MODEL,
        instructions=SYSTEM,
        input=prompt
    )
    return getattr(resp, "output_text", str(resp))

def build_context_block(sections: List[Tuple[str, str]]) -> str:
    """
    sections: list of (title, text)
    """
    parts = []
    for title, text in sections:
        if text:
            parts.append(f"### {title}\n{text}")
    return "\n\n".join(parts)


In [8]:
# RESEND

def send_email_via_resend(to_email: str, subject: str, body_text: str) -> Tuple[bool, str]:
    if not RESEND_API_KEY or not to_email:
        return False, "Missing RESEND_API_KEY or recipient email."
    url = "https://api.resend.com/emails"
    payload = {
        "from": "Agent Demo <onboarding@resend.dev>",  # You can change this if verified
        "to": [to_email],
        "subject": subject,
        "text": body_text
    }
    r = requests.post(url, json=payload, headers={"Authorization": f"Bearer {RESEND_API_KEY}"}, timeout=30)
    if r.ok:
        return True, "Email sent."
    return False, f"Email failed: {r.status_code} {r.text[:200]}"


In [10]:
import gradio as gr
from typing import List, Tuple


from typing import List, Tuple

def answer_pipeline(
    question: str,
    use_pdf: bool,
    use_tavily: bool,
    use_playwright: bool,
    url_override: str,
    send_email: bool,
    email_address: str,
):
    if not question.strip():
        return "Please enter a question.", ""

    sections: List[Tuple[str, str]] = []
    chosen_url = ""
    scraped_text = ""

    # ---------- PDF (page-targeted first, then semantic) ----------
    if use_pdf:
        try:
            page_num = get_page_number_from_question(question)
            if page_num:
                # Pull ONLY from that page
                docs = retrieve_page_docs(page_num, k=max(12, TOP_K))
                if not docs:
                    sections.append(("PDF Context", f"[No chunks found for page {page_num}.]"))
                else:
                    ctx = format_docs_with_citations(docs)
                    sections.append((f"PDF Context (page {page_num})", ctx))
            else:
                # Normal semantic retrieval via LangChain retriever (MMR)
                retriever = vectorstore.as_retriever(
                    search_type="mmr",
                    search_kwargs={"k": TOP_K, "fetch_k": max(20, 3*TOP_K)}
                )
                docs = retriever.get_relevant_documents(question)
                if docs:
                    ctx = format_docs_with_citations(docs)
                    sections.append(("PDF Context", ctx))
                else:
                    sections.append(("PDF Context", "[No relevant chunks retrieved.]"))
        except Exception as e:
            sections.append(("PDF Error", f"[PDF retrieval error] {e}"))

    # ---------- Tavily (optional) ----------
    if use_tavily:
        try:
            t_results = tavily_search(question, max_results=5)
            if t_results:
                sections.append(("Tavily Snippets", summarize_tavily_results(t_results)))
                chosen_url = t_results[0].get("url") or chosen_url
        except Exception as e:
            sections.append(("Tavily Error", f"[Tavily error] {e}"))

    # ---------- Playwright (optional) ----------
    target_url = (url_override or "").strip() or chosen_url
    if use_playwright and target_url:
        try:
            scraped_text = scrape_page(target_url)
            if scraped_text:
                sections.append(("Playwright Page Text", scraped_text))
        except Exception as e:
            sections.append(("Playwright Error", f"[Scrape error] {e}"))

    # ---------- Build one prompt & ask once ----------
    context_block = build_context_block(sections).strip()
    if not context_block:
        prompt = f"User question: {question}\n\nAnswer helpfully and concisely."
    else:
        prompt = (
            f"{context_block}\n\n"
            f"User question: {question}\n\n"
            f"Answer concisely. If you use info from above, reference the section title "
            f"(e.g., PDF Context, Tavily Snippets, Playwright Page Text) and cite pages like (p. 71)."
        )

    try:
        final = call_gpt5(prompt)  # call GPT
    except Exception as e:
        final = f"[Model error] {e}"

    # ---------- Email (optional) ----------
    email_status = ""
    if send_email and email_address.strip():
        ok, msg = send_email_via_resend(email_address.strip(), "Agent Answer", final)
        email_status = msg

    return final, email_status


with gr.Blocks(title="Local Agent Demo (LangChain)") as demo:
    gr.Markdown("## 🧪 Local Agent Demo")
    with gr.Row():
        question = gr.Textbox(label="Ask a question", placeholder="Ask about the PDF or anything else…")
    with gr.Row():
        use_pdf = gr.Checkbox(value=True, label="Use PDF (LangChain retrieval)")
        use_tavily = gr.Checkbox(value=False, label="Use Tavily web search")
        use_playwright = gr.Checkbox(value=False, label="Use Playwright to scrape a page")
        send_email = gr.Checkbox(value=False, label="Email Answer")
    with gr.Row():
        url_override = gr.Textbox(label="URL to scrape (optional):")
        email_address = gr.Textbox(label="Email Answer to:", placeholder="you@example.com")
    with gr.Row():
        submit = gr.Button("Ask Me")
    with gr.Row():
        answer_box = gr.Textbox(label="Answer", lines=10)
    with gr.Row():
        email_status = gr.Textbox(label="Email status", lines=2)

    submit.click(
        fn= answer_pipeline,
        inputs=[question, use_pdf, use_tavily, use_playwright, url_override, send_email, email_address,],
        outputs=[answer_box, email_status]
    )

demo.launch(share=False)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
